# Data Cleaning

This notebook is used to:

- Sort the csv file by image_path for better cleaning

- Annotate to write the caption on the image to easier spot falsely labeled image (via files explorer or any interface you know :D )

In [1]:
INPUT_CSV_FILE = "SYN_FIRE.csv"
IMAGE_DIR = "SYN_FIRE"
OUTPUT_DIR = "fire_annotated"

In [2]:
import pandas as pd
df = pd.read_csv(INPUT_CSV_FILE)
df

,image_path,label,caption,explanation
0,SYN_FIRE/178.png,controlled fire,The image shows the interior of a large wareho...,The presence of a small flame on one of the sh...
1,SYN_FIRE/359.png,controlled fire,The image depicts a dimly lit warehouse with s...,The flame is small and confined to a specific ...
2,SYN_FIRE/147.png,dangerous fire,The image shows a warehouse aisle with shelves...,The presence of flames coming from the boxes i...
3,SYN_FIRE/340.png,dangerous fire,The image depicts a large industrial warehouse...,"The fire is out of control, with flames rising..."
4,SYN_FIRE/525.png,dangerous fire,The image shows an industrial warehouse with s...,The presence of flames and the dim lighting su...
...,...,...,...,...
995,SYN_FIRE/281.png,dangerous fire,The image depicts the interior of a large ware...,The presence of thick smoke rising from the fi...
996,SYN_FIRE/326.png,dangerous fire,The image shows an industrial warehouse with h...,The presence of two large flames or plumes of ...
997,SYN_FIRE/98.png,no fire,The image shows a dimly lit warehouse with sta...,"There are no visible signs of fire, smoke, or ..."
998,SYN_FIRE/196.png,no fire,The image shows the interior of a large wareho...,"There are no visible signs of fire, smoke, or ..."


In [4]:
import re

# Extract the number from the image_path (e.g., image123.jpg → 123)
df['sort_key'] = df['image_path'].apply(lambda x: int(re.search(r'(\d+)', x).group(1)))

df_sorted = df.sort_values(by='sort_key')
df_sorted = df_sorted.drop(columns=['sort_key'])

df_sorted.to_csv(INPUT_CSV_FILE, index=False)
print(f"Numerical sorting complete. Overwrited as {INPUT_CSV_FILE}")

Numerical sorting complete. Overwrited as SYN_FIRE.csv


In [ ]:
import os
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load larger font
try:
    font = ImageFont.truetype("arial.ttf", size=30)
except:
    font = ImageFont.load_default()

# Annotate images
for _, row in df.iterrows():
    image_path = row['image_path']
    label = row['label']

    try:
        img = Image.open(image_path).convert("RGB")
        draw = ImageDraw.Draw(img)

        # Compose text
        text = f"{label.upper()}"

        # Measure text size
        bbox = draw.textbbox((0, 0), text, font=font)
        text_width = bbox[2] - bbox[0]
        text_height = bbox[3] - bbox[1]
        padding = 20
        new_height = img.height + text_height + 2 * padding

        # Create new image with extra space at bottom
        annotated_img = Image.new("RGB", (img.width, new_height), "white")
        annotated_img.paste(img, (0, 0))

        # Draw centered text
        draw = ImageDraw.Draw(annotated_img)
        x = (img.width - text_width) // 2
        y = img.height + padding
        draw.text((x, y), text, fill="black", font=font)

        # Save result
        fname = os.path.basename(image_path)
        annotated_img.save(os.path.join(OUTPUT_DIR, fname))
        print(f"✅ Annotated {fname}")
    except Exception as e:
        print(f"❌ Failed to annotate {image_path}: {e}")

print(f"Annotations finished, please check your {OUTPUT_DIR} folder")
